# 🛶 Kayak — Data Engineering : Météo, Hôtels & Pipeline AWS

> *Construire un pipeline de données complet : API météo → Scraping Booking.com → AWS S3 → AWS RDS*

**Périmètre :** 35 villes touristiques françaises  
**Stack :** Nominatim · OpenWeatherMap · Booking.com (scraping) · AWS S3 · AWS RDS (PostgreSQL)

---

## 1. 📦 Imports & Configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import boto3
import psycopg2
import plotly.express as px
import plotly.graph_objects as go
from bs4 import BeautifulSoup
from sqlalchemy import create_engine
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Configuration AWS ──────────────────────────────────────────────
import os
from dotenv import load_dotenv
load_dotenv()

AWS_ACCESS_KEY    = os.getenv('AWS_ACCESS_KEY_ID')
AWS_SECRET_KEY    = os.getenv('AWS_SECRET_ACCESS_KEY')
AWS_REGION        = os.getenv('AWS_DEFAULT_REGION', 'eu-west-3')
S3_BUCKET         = os.getenv('S3_BUCKET_NAME')
OPENWEATHER_KEY   = os.getenv('OPENWEATHER_API_KEY')
RDS_HOST          = os.getenv('RDS_HOST')
RDS_PORT          = os.getenv('RDS_PORT', '5432')
RDS_DB            = os.getenv('RDS_DB')
RDS_USER          = os.getenv('RDS_USER')
RDS_PASSWORD      = os.getenv('RDS_PASSWORD')

print("✅ Configuration chargée")

# 35 villes touristiques françaises
CITIES = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy", "Grenoble",
    "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne",
    "Ariege", "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]
print(f"Nombre de villes : {len(CITIES)}")

## 2. 🌍 Coordonnées GPS via Nominatim

In [ ]:
def get_coordinates(city):
    """Récupère lat/lon d'une ville via l'API Nominatim (OpenStreetMap)."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": f"{city}, France", "format": "json", "limit": 1}
    headers = {"User-Agent": "KayakProject/1.0 (jedha-student@example.com)"}
    
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        data = r.json()
        if data:
            return {
                "city": city,
                "city_id": city.lower().replace(" ", "_"),
                "lat": float(data[0]["lat"]),
                "lon": float(data[0]["lon"])
            }
    except Exception as e:
        print(f"  ⚠️ Erreur pour {city}: {e}")
    return None

# Récupération des coordonnées
cities_data = []
for city in CITIES:
    result = get_coordinates(city)
    if result:
        cities_data.append(result)
        print(f"✅ {city:35s} → lat: {result['lat']:.4f}, lon: {result['lon']:.4f}")
    time.sleep(1)  # Respecter le rate limit Nominatim (1 req/sec)

df_cities = pd.DataFrame(cities_data)
print(f"\n{len(df_cities)}/{len(CITIES)} villes géocodées")
df_cities.head()

## 3. 🌤️ Données Météo via OpenWeatherMap

In [ ]:
def get_weather(lat, lon, api_key):
    """Récupère les prévisions météo sur 7 jours via OpenWeatherMap One Call API."""
    url = "https://api.openweathermap.org/data/3.0/onecall"
    params = {
        "lat": lat, "lon": lon,
        "appid": api_key,
        "units": "metric",
        "exclude": "current,minutely,hourly,alerts"
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        return r.json()
    except Exception as e:
        print(f"  ⚠️ Erreur météo: {e}")
        return None

def parse_weather(city_id, city_name, weather_data):
    """Parse les données météo journalières sur 7 jours."""
    records = []
    if not weather_data or 'daily' not in weather_data:
        return records
    
    for day in weather_data['daily'][:7]:
        records.append({
            "city_id":       city_id,
            "city":          city_name,
            "date":          datetime.fromtimestamp(day['dt']).strftime('%Y-%m-%d'),
            "temp_min":      day['temp']['min'],
            "temp_max":      day['temp']['max'],
            "temp_day":      day['temp']['day'],
            "humidity":      day['humidity'],
            "wind_speed":    day['wind_speed'],
            "pop":           day.get('pop', 0),          # Probabilité de pluie
            "rain_mm":       day.get('rain', 0),          # Volume de pluie (mm)
            "weather_desc":  day['weather'][0]['description'],
            "weather_icon":  day['weather'][0]['icon']
        })
    return records

# Récupération météo pour toutes les villes
all_weather = []
for _, row in df_cities.iterrows():
    print(f"⛅ {row['city']:35s}", end=" → ")
    data = get_weather(row['lat'], row['lon'], OPENWEATHER_KEY)
    records = parse_weather(row['city_id'], row['city'], data)
    all_weather.extend(records)
    print(f"{len(records)} jours récupérés")
    time.sleep(0.5)

df_weather = pd.DataFrame(all_weather)
print(f"\n✅ {len(df_weather)} enregistrements météo ({len(df_cities)} villes × 7 jours)")
df_weather.head(10)

## 4. 🏆 Score Météo & Top 5 Destinations

In [ ]:
def compute_weather_score(df):
    """
    Score météo sur 100 basé sur :
    - Température journalière (idéale : 20-25°C)
    - Probabilité de pluie (plus c'est bas, mieux c'est)
    - Humidité (idéale : 40-60%)
    - Vitesse du vent (plus c'est bas, mieux c'est)
    """
    df = df.copy()
    
    # Score température (gaussienne centrée sur 22°C)
    df['score_temp']   = np.exp(-((df['temp_day'] - 22) ** 2) / 50)
    
    # Score pluie (1 = pas de pluie, 0 = pluie certaine)
    df['score_rain']   = 1 - df['pop']
    
    # Score humidité
    df['score_humid']  = 1 - abs(df['humidity'] - 50) / 50
    
    # Score vent (optimal < 5 km/h)
    df['score_wind']   = np.exp(-df['wind_speed'] / 10)
    
    # Score global pondéré
    df['weather_score'] = (
        0.40 * df['score_temp'] +
        0.35 * df['score_rain'] +
        0.15 * df['score_humid'] +
        0.10 * df['score_wind']
    ) * 100
    
    return df

df_weather = compute_weather_score(df_weather)

# Score moyen sur 7 jours par ville
df_city_scores = df_weather.groupby(['city_id','city']).agg(
    avg_temp    = ('temp_day', 'mean'),
    avg_rain_mm = ('rain_mm', 'mean'),
    avg_pop     = ('pop', 'mean'),
    avg_humidity= ('humidity', 'mean'),
    weather_score = ('weather_score', 'mean')
).reset_index().sort_values('weather_score', ascending=False)

print("🏆 Top 10 destinations météo :")
print(df_city_scores[['city','avg_temp','avg_rain_mm','avg_pop','weather_score']].head(10).round(2).to_string(index=False))

TOP5_CITIES = df_city_scores.head(5)['city'].tolist()
print(f"\n✅ Top 5 : {TOP5_CITIES}")

## 5. 🗺️ Carte des 5 Meilleures Destinations

In [ ]:
df_map = df_city_scores.merge(df_cities[['city','lat','lon']], on='city', how='left')

fig_dest = px.scatter_mapbox(
    df_map,
    lat="lat", lon="lon",
    size="weather_score",
    color="weather_score",
    color_continuous_scale="RdYlGn",
    hover_name="city",
    hover_data={"avg_temp": ":.1f", "avg_pop": ":.1%", "weather_score": ":.1f",
                "lat": False, "lon": False},
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    mapbox_style="carto-positron",
    title="🌤️ Score Météo — 35 Destinations Françaises (7 prochains jours)",
    size_max=30,
    labels={"weather_score": "Score météo", "avg_temp": "Temp. moy (°C)", "avg_pop": "Prob. pluie"}
)

# Annoter le Top 5
top5_df = df_map[df_map['city'].isin(TOP5_CITIES)]
fig_dest.add_trace(go.Scattermapbox(
    lat=top5_df['lat'], lon=top5_df['lon'],
    mode='markers+text',
    marker=dict(size=18, color='gold', symbol='star'),
    text=["⭐ " + c for c in top5_df['city']],
    textposition='top right',
    name='Top 5 destinations',
    textfont=dict(size=11, color='black')
))

fig_dest.update_layout(height=600, margin=dict(t=50,b=0,l=0,r=0))
fig_dest.show()
print(f"\n🌟 Top 5 destinations : {', '.join(TOP5_CITIES)}")

## 6. 🏨 Scraping Hôtels Booking.com

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
}

def scrape_booking(city, n_hotels=20):
    """Scrape les hôtels Booking.com pour une ville donnée."""
    city_encoded = city.replace(" ", "+")
    url = f"https://www.booking.com/searchresults.fr.html?ss={city_encoded}&checkin=2025-07-14&checkout=2025-07-21&group_adults=2&no_rooms=1&order=review_score_and_price"
    
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(r.content, 'html.parser')
        hotels = []
        
        # Sélecteurs Booking.com (susceptibles de changer)
        hotel_cards = soup.find_all('div', {'data-testid': 'property-card'})
        
        for card in hotel_cards[:n_hotels]:
            try:
                name_el    = card.find('div', {'data-testid': 'title'})
                score_el   = card.find('div', {'data-testid': 'review-score'})
                desc_el    = card.find('div', {'data-testid': 'property-card-unit-configuration'})
                url_el     = card.find('a', {'data-testid': 'property-card-desktop-single-image'})
                lat_el     = card.get('data-lat')
                lon_el     = card.get('data-lon')
                
                hotels.append({
                    "city":         city,
                    "city_id":      city.lower().replace(" ", "_"),
                    "hotel_name":   name_el.text.strip() if name_el else None,
                    "booking_url":  "https://www.booking.com" + url_el['href'] if url_el else None,
                    "score":        float(score_el.text.strip().replace(',','.')) if score_el else None,
                    "description":  desc_el.text.strip() if desc_el else None,
                    "lat":          float(lat_el) if lat_el else None,
                    "lon":          float(lon_el) if lon_el else None,
                })
            except Exception:
                continue
        
        return hotels
    
    except Exception as e:
        print(f"  ⚠️ Erreur scraping {city}: {e}")
        return []

# Scraping pour le Top 5 destinations
all_hotels = []
for city in TOP5_CITIES:
    print(f"🏨 Scraping {city:30s}", end=" → ")
    hotels = scrape_booking(city, n_hotels=20)
    all_hotels.extend(hotels)
    print(f"{len(hotels)} hôtels trouvés")
    time.sleep(3)  # Pause entre les requêtes

df_hotels = pd.DataFrame(all_hotels)
df_hotels = df_hotels.dropna(subset=['hotel_name'])
print(f"\n✅ {len(df_hotels)} hôtels scrapés au total")
df_hotels.head()

## 7. 🗺️ Carte des 20 Hôtels les Mieux Notés

In [ ]:
# Top 20 hôtels toutes villes confondues
df_top_hotels = df_hotels.dropna(subset=['score','lat','lon']).sort_values('score', ascending=False).head(20)

fig_hotels = px.scatter_mapbox(
    df_top_hotels,
    lat="lat", lon="lon",
    color="city",
    size="score",
    size_max=20,
    hover_name="hotel_name",
    hover_data={"score": True, "city": True, "lat": False, "lon": False},
    zoom=5,
    center={"lat": 46.5, "lon": 2.5},
    mapbox_style="carto-positron",
    title="🏨 Top 20 Hôtels les Mieux Notés — Meilleures Destinations",
    labels={"score": "Note Booking", "city": "Ville"}
)
fig_hotels.update_layout(height=600, margin=dict(t=50,b=0,l=0,r=0))
fig_hotels.show()

print("🏆 Top 10 hôtels :")
print(df_top_hotels[['hotel_name','city','score']].head(10).to_string(index=False))

## 8. 💾 Sauvegarde locale des données

In [ ]:
# Sauvegarde CSV
df_weather.to_csv('weather_data.csv', index=False)
df_hotels.to_csv('hotels_data.csv', index=False)
df_city_scores.to_csv('city_scores.csv', index=False)
df_cities.to_csv('cities_coordinates.csv', index=False)

print("✅ Fichiers CSV sauvegardés :")
print(f"  - weather_data.csv      ({len(df_weather)} lignes)")
print(f"  - hotels_data.csv       ({len(df_hotels)} lignes)")
print(f"  - city_scores.csv       ({len(df_city_scores)} lignes)")
print(f"  - cities_coordinates.csv ({len(df_cities)} lignes)")

## 9. ☁️ Chargement sur AWS S3 (Data Lake)

In [ ]:
import io

s3 = boto3.client(
    's3',
    aws_access_key_id     = AWS_ACCESS_KEY,
    aws_secret_access_key = AWS_SECRET_KEY,
    region_name           = AWS_REGION
)

def upload_df_to_s3(df, bucket, key):
    """Upload un DataFrame pandas vers S3 au format CSV."""
    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    print(f"  ✅ s3://{bucket}/{key} ({len(df)} lignes)")

print("📤 Upload vers S3...")
upload_df_to_s3(df_cities,      S3_BUCKET, "raw/cities_coordinates.csv")
upload_df_to_s3(df_weather,     S3_BUCKET, "raw/weather_data.csv")
upload_df_to_s3(df_hotels,      S3_BUCKET, "raw/hotels_data.csv")
upload_df_to_s3(df_city_scores, S3_BUCKET, "processed/city_scores.csv")

print("\n✅ Data Lake S3 alimenté !")

## 10. 🗄️ ETL : S3 → AWS RDS (PostgreSQL)

In [ ]:
# ── Connexion RDS ──────────────────────────────────────────────────
engine = create_engine(
    f"postgresql+psycopg2://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DB}"
)

print("🔌 Connexion à RDS PostgreSQL...")

# ── ETL : lecture depuis S3 ─────────────────────────────────────────
def read_s3_csv(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

df_cities_clean  = read_s3_csv(S3_BUCKET, "raw/cities_coordinates.csv")
df_weather_clean = read_s3_csv(S3_BUCKET, "raw/weather_data.csv")
df_hotels_clean  = read_s3_csv(S3_BUCKET, "raw/hotels_data.csv")

# ── Nettoyage avant chargement ──────────────────────────────────────
df_hotels_clean = df_hotels_clean.dropna(subset=['hotel_name'])
df_weather_clean['date'] = pd.to_datetime(df_weather_clean['date'])

# ── Chargement dans RDS ─────────────────────────────────────────────
print("\n📥 Chargement dans RDS...")

df_cities_clean.to_sql('cities', engine, if_exists='replace', index=False)
print(f"  ✅ Table 'cities'  : {len(df_cities_clean)} lignes")

df_weather_clean.to_sql('weather', engine, if_exists='replace', index=False)
print(f"  ✅ Table 'weather' : {len(df_weather_clean)} lignes")

df_hotels_clean.to_sql('hotels', engine, if_exists='replace', index=False)
print(f"  ✅ Table 'hotels'  : {len(df_hotels_clean)} lignes")

print("\n✅ Data Warehouse RDS alimenté !")

## 11. 🔍 Requêtes SQL de Vérification

In [ ]:
# Vérification des données dans RDS
with engine.connect() as conn:
    print("📊 Aperçu des tables RDS :")
    
    print("\n-- Top 5 villes par score météo --")
    q1 = pd.read_sql("""
        SELECT city, ROUND(AVG(weather_score)::numeric, 2) as avg_score,
               ROUND(AVG(temp_day)::numeric, 1) as avg_temp,
               ROUND(AVG(pop)::numeric, 2) as avg_rain_proba
        FROM weather
        GROUP BY city
        ORDER BY avg_score DESC
        LIMIT 5
    """, conn)
    print(q1.to_string(index=False))
    
    print("\n-- Top 10 hôtels par note --")
    q2 = pd.read_sql("""
        SELECT hotel_name, city, score
        FROM hotels
        WHERE score IS NOT NULL
        ORDER BY score DESC
        LIMIT 10
    """, conn)
    print(q2.to_string(index=False))

## 12. 📝 Conclusion

### Pipeline complet réalisé

```
Nominatim API          → Coordonnées GPS (35 villes)
        ↓
OpenWeatherMap API     → Prévisions météo 7 jours
        ↓
Score météo maison     → Classement des destinations
        ↓
Booking.com scraping   → Hôtels (Top 5 destinations)
        ↓
AWS S3 (Data Lake)     → Stockage raw + processed CSV
        ↓
AWS RDS (PostgreSQL)   → Data Warehouse requêtable
        ↓
Plotly Maps            → 2 cartes interactives
```

### Résultats

- **35 villes** géocodées et analysées météorologiquement
- **7 jours** de prévisions par ville avec score composite
- **Top 5 destinations** identifiées selon critères météo
- **Top 20 hôtels** scrapés sur Booking.com avec notes et coordonnées
- **Data Lake S3** alimenté avec les données brutes et traitées
- **Data Warehouse RDS** prêt pour les analyses marketing Kayak
